# 향수 데이터셋 구조 확인

## 분석 목적

이 노트북은 추천 알고리즘 설계나 데이터 전처리에 앞서 데이터셋의 실제 구조를 정확히 이해하기 위한 것입니다.

- `perfumes.csv`와 `perfumes.jsonl`의 실제 구조 확인
- 두 파일의 레코드 수 비교
- JSONL의 중첩 구조 확인
- `SCHEMA.md`의 설명과 실제 데이터가 일치하는지 확인
- 이후 EDA에서 어떤 파일을 기준 데이터로 사용할지 판단

> 이 단계에서는 결측치 분석, 시각화, 데이터 전처리, 추천 알고리즘 구현을 수행하지 않습니다.

## 1. 라이브러리 불러오기

데이터 구조 확인과 파일 읽기에 필요한 최소 라이브러리만 불러옵니다.

In [1]:
import pandas as pd
import json
import pathlib
import os

## 2. 현재 작업 경로와 파일 확인

현재 작업 경로를 확인하고, 분석 대상 세 파일의 존재 여부와 크기를 MB 단위로 출력합니다.

In [2]:
work_dir = pathlib.Path(os.getcwd())
file_names = ["perfumes.csv", "perfumes.jsonl", "SCHEMA.md"]

print(f"현재 작업 경로: {work_dir}")
print()

for file_name in file_names:
    file_path = work_dir / file_name
    exists = file_path.is_file()
    size_mb = file_path.stat().st_size / (1024 ** 2) if exists else None
    size_text = f"{size_mb:.2f} MB" if exists else "확인 불가"
    print(f"{file_name:<16} | 존재: {exists!s:<5} | 용량: {size_text}")

현재 작업 경로: c:\Users\dyftj\OneDrive\바탕 화면\향수_데이터

perfumes.csv     | 존재: True  | 용량: 121.43 MB
perfumes.jsonl   | 존재: True  | 용량: 485.94 MB
SCHEMA.md        | 존재: True  | 용량: 0.01 MB


## 3. SCHEMA.md 앞부분 확인

스키마 문서를 UTF-8로 읽고 앞 60줄을 출력합니다. 원본 파일은 수정하지 않습니다.

In [3]:
schema_path = work_dir / "SCHEMA.md"
schema_text = schema_path.read_text(encoding="utf-8")
schema_preview = "\n".join(schema_text.splitlines()[:60])

print(schema_preview)

# Dump schema — `perfumes.jsonl.zst`

One JSON object per line, zstd-compressed. The whole corpus, every perfume, unfiltered.

```sh
zstd -dc perfumes.jsonl.zst | jq        # browse
zstd -dc perfumes.jsonl.zst | wc -l     # count
```

This dump is the complete corpus. `perfumes.db` is a trimmed SQLite **showcase** built from a subset of it — vote-filtered, with the similar-perfume carousels capped and a few fields dropped — so use this dump if you want everything. Run `.schema` on the db for its shape.

## A record

```jsonc
{
  "id": 9828,
  "slug": "Creed/Aventus",
  "url": "https://www.fragrantica.com/perfume/Creed/Aventus-9828.html",
  "name": "Aventus", "brand": "Creed", "year": 2010,
  "collection": "Aventus", "gender": "male",
  "description": "Aventus by Creed is …",
  "picture":   "https://fimgs.net/mdimg/perfume/375x500.9828.jpg",
  "thumbnail": "https://fimgs.net/mdimg/perfume/m.9828.jpg",

  "perfumers": [ { "name": "Erwin Creed", "slug": "Erwin_Creed", "image_id": 865 } ],

## 4. CSV 구조 확인

`perfumes.csv`를 pandas DataFrame으로 읽고 크기, 컬럼, 자료형 및 첫 3개 레코드를 확인합니다.

In [4]:
csv_path = work_dir / "perfumes.csv"
df = pd.read_csv(csv_path, low_memory=False)

print(f"shape: {df.shape}")
print(f"columns ({len(df.columns)}개):")
print(df.columns.tolist())

shape: (131930, 59)
columns (59개):
['id', 'slug', 'name', 'brand', 'year', 'collection', 'gender', 'url', 'rating_avg', 'vote_count', 'rating_b1', 'rating_b2', 'rating_b3', 'rating_b4', 'rating_b5', 'longevity_avg', 'longevity_b1', 'longevity_b2', 'longevity_b3', 'longevity_b4', 'longevity_b5', 'sillage_avg', 'sillage_b1', 'sillage_b2', 'sillage_b3', 'sillage_b4', 'price_value_avg', 'price_value_b1', 'price_value_b2', 'price_value_b3', 'price_value_b4', 'price_value_b5', 'have', 'had', 'want', 'perceived_female', 'perceived_female_leaning', 'perceived_unisex', 'perceived_male_leaning', 'perceived_male', 'winter', 'spring', 'summer', 'autumn', 'day', 'night', 'people', 'magnitude', 'compound_magnitude', 'recent_magnitude', 'last_comment_at', 'scraped_at', 'accords', 'notes_top', 'notes_middle', 'notes_base', 'notes_flat', 'perfumers', 'description']


### CSV 컬럼별 자료형 확인

`df.info()`로 각 컬럼의 자료형과 non-null 개수를 확인합니다. 이 단계에서는 결측치를 별도로 분석하지 않습니다.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 131930 entries, 0 to 131929
Data columns (total 59 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   id                        131930 non-null  int64  
 1   slug                      131930 non-null  str    
 2   name                      131930 non-null  str    
 3   brand                     131930 non-null  str    
 4   year                      107797 non-null  float64
 5   collection                55867 non-null   str    
 6   gender                    131930 non-null  str    
 7   url                       131930 non-null  str    
 8   rating_avg                131930 non-null  float64
 9   vote_count                131930 non-null  int64  
 10  rating_b1                 131930 non-null  int64  
 11  rating_b2                 131930 non-null  int64  
 12  rating_b3                 131930 non-null  int64  
 13  rating_b4                 131930 non-null  int64  
 14 

### CSV 첫 3개 레코드 확인

실제 값이 어떤 형태로 저장되어 있는지 첫 3행만 확인합니다.

In [6]:
df.head(3)

,id,slug,name,brand,year,collection,gender,url,rating_avg,vote_count,...,recent_magnitude,last_comment_at,scraped_at,accords,notes_top,notes_middle,notes_base,notes_flat,perfumers,description
0,1,Azzaro/Orange-Tonic,Orange Tonic,Azzaro,NaN,TONIC,female,https://www.fragrantica.com/perfume/Azzaro/Ora...,3.4421,95,...,77,1775717654,1780102667,citrus:100|fresh spicy:47|green:41|sweet:33|ar...,Orange|Bergamot|Basil|Grapefruit|Green Notes|M...,Honeysuckle|Blueberry|Flowers|Lily-of-the-Vall...,Honey|Amber|Cedar|Musk,NaN,Nathalie Feisthauer,Orange Tonic by Azzaro is a Floral Green fragr...
1,3,Givenchy/Amarige,Amarige,Givenchy,1991.0,AMARIGE,female,https://www.fragrantica.com/perfume/Givenchy/A...,3.9053,9072,...,3260,1777937291,1780064964,white floral:100|sweet:52|yellow floral:47|woo...,Orange Blossom|Peach|Plum|Neroli|Brazilian Ros...,Tuberose|Mimosa|Gardenia|Ylang-Ylang|Jasmine|B...,Sandalwood|Amber|Woody Notes|Vanilla|Musk|Tonk...,NaN,Dominique Ropion,Amarige by Givenchy is a Floral fragrance for ...
2,4,Givenchy/Organza,Organza,Givenchy,1996.0,ORGANZA BY GIVENCHY,female,https://www.fragrantica.com/perfume/Givenchy/O...,3.9587,11565,...,3401,1778091134,1780064758,white floral:100|fresh spicy:33|woody:30|vanil...,Nutmeg|Gardenia|African Orange Flower|Green No...,Tuberose|Jasmine|Honeysuckle|Iris|Peony|Mace,Vanilla|Amber|Woodsy Notes|Guaiac Wood|Virgini...,NaN,Sophie Labbé,Organza by Givenchy is a Oriental Floral fragr...


## 5. JSONL 첫 번째 레코드 확인

JSONL 전체를 DataFrame으로 읽지 않고 첫 번째 줄만 `json.loads()`로 파싱합니다. 최상위 키, 각 값의 Python 자료형, 기본 정보를 확인합니다.

In [7]:
jsonl_path = work_dir / "perfumes.jsonl"

with jsonl_path.open("r", encoding="utf-8") as file:
    first_record = json.loads(file.readline())

print("최상위 key 목록:")
print(list(first_record.keys()))

print("\n각 key의 Python 자료형:")
for key, value in first_record.items():
    print(f"{key:<18}: {type(value).__name__}")

basic_fields = ["id", "name", "brand", "year", "gender"]
basic_info = {field: first_record.get(field) for field in basic_fields}

print("\n첫 번째 향수 레코드의 기본 정보:")
print(json.dumps(basic_info, ensure_ascii=False, indent=2))

최상위 key 목록:
['id', 'slug', 'url', 'name', 'brand', 'year', 'collection', 'gender', 'description', 'picture', 'thumbnail', 'perfumers', 'accords', 'notes', 'rating', 'longevity', 'sillage', 'price_value', 'relation', 'community_gender', 'seasons', 'daypart', 'people', 'ai_summary', 'similar', 'popularity', 'meta']

각 key의 Python 자료형:
id                : int
slug              : str
url               : str
name              : str
brand             : str
year              : NoneType
collection        : str
gender            : str
description       : str
picture           : str
thumbnail         : str
perfumers         : list
accords           : list
notes             : dict
rating            : dict
longevity         : dict
sillage           : dict
price_value       : dict
relation          : dict
community_gender  : dict
seasons           : dict
daypart           : dict
people            : int
ai_summary        : dict
similar           : dict
popularity        : dict
meta              : di

## 6. JSONL 앞 5개 레코드의 중첩 구조 확인

앞 5줄만 파싱한 뒤 지정된 중첩 필드의 자료형, 하위 키, 리스트 길이와 첫 항목 구조를 요약합니다. 큰 리스트의 모든 값을 출력하지 않아 구조를 읽기 쉽게 유지합니다.

In [8]:
nested_fields = [
    "perfumers",
    "accords",
    "notes",
    "rating",
    "longevity",
    "sillage",
    "price_value",
    "relation",
    "community_gender",
    "seasons",
    "daypart",
    "ai_summary",
    "similar",
    "popularity",
]

records_5 = []
with jsonl_path.open("r", encoding="utf-8") as file:
    for _ in range(5):
        line = file.readline()
        if not line:
            break
        records_5.append(json.loads(line))

def describe_structure(value):
    if isinstance(value, dict):
        return {key: describe_structure(item) for key, item in value.items()}
    if isinstance(value, list):
        return {
            "type": "list",
            "length": len(value),
            "item_structure": describe_structure(value[0]) if value else None,
        }
    return type(value).__name__

for index, record in enumerate(records_5, start=1):
    print("=" * 80)
    print(f"레코드 {index}: id={record.get('id')}, name={record.get('name')}")
    structure = {
        field: describe_structure(record.get(field))
        for field in nested_fields
    }
    print(json.dumps(structure, ensure_ascii=False, indent=2))

레코드 1: id=1, name=Orange Tonic
{
  "perfumers": {
    "type": "list",
    "length": 1,
    "item_structure": {
      "name": "str",
      "slug": "str",
      "image_id": "int"
    }
  },
  "accords": {
    "type": "list",
    "length": 8,
    "item_structure": {
      "name": "str",
      "strength": "int"
    }
  },
  "notes": {
    "tiered": {
      "top": {
        "type": "list",
        "length": 7,
        "item_structure": {
          "name": "str",
          "slug": "str",
          "image_id": "int"
        }
      },
      "middle": {
        "type": "list",
        "length": 5,
        "item_structure": {
          "name": "str",
          "slug": "str",
          "image_id": "int"
        }
      },
      "base": {
        "type": "list",
        "length": 4,
        "item_structure": {
          "name": "str",
          "slug": "str",
          "image_id": "int"
        }
      }
    },
    "flat": {
      "type": "list",
      "length": 0,
      "item_structure": null
  

## 7. SCHEMA.md 설명과 실제 JSONL 구조 비교

문서에 제시된 주요 최상위 필드가 앞 5개 실제 레코드에 존재하는지 확인하고, 핵심 중첩 객체의 하위 키를 비교합니다.

In [9]:
schema_top_level_fields = [
    "id", "slug", "url", "name", "brand", "year",
    "collection", "gender", "description", "picture", "thumbnail",
    "perfumers", "accords", "notes", "rating", "longevity",
    "sillage", "price_value", "relation", "community_gender",
    "seasons", "daypart", "people", "ai_summary", "similar",
    "popularity", "meta",
]

expected_dict_keys = {
    "notes": {"tiered", "flat"},
    "rating": {"average", "histogram"},
    "longevity": {"average", "histogram"},
    "sillage": {"average", "histogram"},
    "price_value": {"average", "histogram"},
    "relation": {"have", "had", "want"},
    "community_gender": {"female", "female_leaning", "unisex", "male_leaning", "male"},
    "seasons": {"winter", "spring", "summer", "autumn"},
    "daypart": {"day", "night"},
    "ai_summary": {"pros", "cons"},
    "similar": {"reminds_me_of", "also_liked"},
    "popularity": {"magnitude", "compound_magnitude", "recent_magnitude", "last_comment_at"},
}

for index, record in enumerate(records_5, start=1):
    missing_top_level = [
        field for field in schema_top_level_fields if field not in record
    ]
    print(f"레코드 {index} - 누락된 주요 최상위 필드: {missing_top_level or '없음'}")

print("\n첫 번째 레코드의 핵심 중첩 키 비교:")
for field, expected_keys in expected_dict_keys.items():
    value = first_record.get(field)
    actual_keys = set(value.keys()) if isinstance(value, dict) else set()
    print(
        f"{field:<18} | 문서 기준 키 포함: {expected_keys.issubset(actual_keys)} "
        f"| 실제 키: {sorted(actual_keys)}"
    )

레코드 1 - 누락된 주요 최상위 필드: 없음
레코드 2 - 누락된 주요 최상위 필드: 없음
레코드 3 - 누락된 주요 최상위 필드: 없음
레코드 4 - 누락된 주요 최상위 필드: 없음
레코드 5 - 누락된 주요 최상위 필드: 없음

첫 번째 레코드의 핵심 중첩 키 비교:
notes              | 문서 기준 키 포함: True | 실제 키: ['flat', 'tiered']
rating             | 문서 기준 키 포함: True | 실제 키: ['average', 'histogram']
longevity          | 문서 기준 키 포함: True | 실제 키: ['average', 'histogram']
sillage            | 문서 기준 키 포함: True | 실제 키: ['average', 'histogram']
price_value        | 문서 기준 키 포함: True | 실제 키: ['average', 'histogram']
relation           | 문서 기준 키 포함: True | 실제 키: ['had', 'have', 'want']
community_gender   | 문서 기준 키 포함: True | 실제 키: ['female', 'female_leaning', 'male', 'male_leaning', 'unisex']
seasons            | 문서 기준 키 포함: True | 실제 키: ['autumn', 'spring', 'summer', 'winter']
daypart            | 문서 기준 키 포함: True | 실제 키: ['day', 'night']
ai_summary         | 문서 기준 키 포함: True | 실제 키: ['cons', 'pros']
similar            | 문서 기준 키 포함: True | 실제 키: ['also_liked', 'reminds_me_of']
popularity         | 문서 기준 키

## 8. JSONL 전체 줄 수 계산

파일을 한꺼번에 메모리에 올리지 않고 한 줄씩 순회하여 전체 줄 수와 비어 있지 않은 레코드 줄 수를 계산합니다.

In [10]:
jsonl_line_count = 0
jsonl_record_count = 0

with jsonl_path.open("r", encoding="utf-8") as file:
    for line in file:
        jsonl_line_count += 1
        if line.strip():
            jsonl_record_count += 1

print(f"JSONL 전체 줄 수: {jsonl_line_count:,}")
print(f"JSONL 레코드 수(빈 줄 제외): {jsonl_record_count:,}")

JSONL 전체 줄 수: 131,930
JSONL 레코드 수(빈 줄 제외): 131,930


## 9. CSV 행 수와 JSONL 레코드 수 비교

CSV의 행 수와 JSONL의 비어 있지 않은 줄 수를 비교하여 두 파일의 레코드 규모가 같은지 확인합니다.

In [11]:
csv_record_count = len(df)
record_count_difference = csv_record_count - jsonl_record_count

print(f"CSV 레코드 수:   {csv_record_count:,}")
print(f"JSONL 레코드 수: {jsonl_record_count:,}")
print(f"차이(CSV - JSONL): {record_count_difference:,}")
print(f"레코드 수 일치 여부: {csv_record_count == jsonl_record_count}")

CSV 레코드 수:   131,930
JSONL 레코드 수: 131,930
차이(CSV - JSONL): 0
레코드 수 일치 여부: True


## 10. 확인 결과 정리 템플릿

위 셀의 실행 결과를 확인한 뒤 아래 빈칸을 직접 채웁니다. 기준 파일 판단 시에는 레코드 수뿐 아니라 원본 중첩 정보의 보존 여부와 이후 분석 목적도 함께 고려합니다.

- **CSV 레코드 수:** 131,930
- **JSONL 레코드 수:** 131,930
- **CSV 컬럼 수:** 59
- **JSONL 주요 중첩 필드:** _[입력]_
- **CSV와 JSONL의 주요 차이:** 0
- **이후 분석에서 사용할 기준 파일 판단:** 둘 다 가능